# SC21 - Grad-CAM Explainability (ConvNeXt-S)
Gradient-weighted class activation maps on the deployed backbone's final 7x7 feature map,
for the **material** head (predicted class) and the **threat** head (total ordinal evidence).
Produces one PNG per showcase item plus a combined grid for the report; items are selected
programmatically (confident successes + the two known-hard classes) so the run is reproducible.

**Attach:** `thesis-common-10c` + the raw image dataset + the SC12 output
(`weights_convnext_small.fb_in22k_ft_in1k.pt`). GPU on. Runtime: ~10-20 min.

In [ ]:
import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| threat levels:', T.NUM_THREAT)
OUT = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# ---- load the deployed classifier ----
import torch
BB = 'convnext_small.fb_in22k_ft_in1k'
wpath = T.find_one(f'weights_{BB}.pt')
model = T.MultiTaskNet(BB).to(T.DEVICE)
model.load_state_dict(torch.load(wpath, map_location=T.DEVICE))
model.eval()
print('loaded', wpath)

ROOT = T.find_image_root()
_, _, df_te = T.load_manifests(ROOT)
df_te = df_te.reset_index(drop=True)

In [ ]:
# ---- Grad-CAM on the ConvNeXt-S final stage (7x7x768 feature map) ----
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

def preprocess_row(r):
    img = T.load_rgb(r['abspath'])
    img = T.object_crop(img, (r['bbox_x'], r['bbox_y'], r['bbox_w'], r['bbox_h']))
    img = T.gray_world(img)
    x = T.apply_tf(T.eval_tf, img).unsqueeze(0).to(T.DEVICE)
    disp = np.array(Image.fromarray(img).resize((T.IMG_SIZE, T.IMG_SIZE)))
    return x, disp

def gradcam(x, head, cls=None):
    model.zero_grad(set_to_none=True)
    with torch.enable_grad():
        f = model.backbone.forward_features(x)          # (1, C, h, w)
        f.retain_grad()
        try:
            pooled = model.backbone.forward_head(f, pre_logits=True)
        except TypeError:
            pooled = model.backbone.forward_head(f)
        z = getattr(model, 'h_' + head)(pooled)
        if head == 'condition':
            score = z[0, 0]
        elif head == 'threat':
            score = z[0].sum()          # total ordinal threat evidence (CORN logits)
        else:
            cls = int(z[0].argmax()) if cls is None else int(cls)
            score = z[0, cls]
        score.backward()
    w = f.grad.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((w * f).sum(1))[0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = np.array(Image.fromarray((cam.detach().cpu().numpy() * 255).astype(np.uint8))
                   .resize((T.IMG_SIZE, T.IMG_SIZE), Image.BILINEAR)) / 255.0
    return cam, z.detach().float().cpu().numpy()[0]

def overlay(disp, cam, alpha=0.45):
    heat = cm.jet(cam)[..., :3]
    return np.clip((1 - alpha) * disp / 255.0 + alpha * heat, 0, 1)

In [ ]:
# ---- pick showcase items programmatically and render panels (one PNG per item + a grid) ----
va = df_te.copy()
# item-level predictions to drive the selection
_, _, te_loader = T.make_loaders(df_te, df_te, df_te)
logits, ys, its = T.evaluate_logits(model, te_loader)
agg, uniq, idx = T.aggregate_by_item(logits, its)
lab = T.item_labels(ys, its, idx)
mpred = T.softmax_np(agg['material']).argmax(1)
correct = mpred == lab['material']

def first_item(mask):
    for i in np.argsort(-T.softmax_np(agg['material']).max(1)):
        if mask[i]: return uniq[i]
    return None

M = {m: k for k, m in enumerate(T.MATERIALS)}
showcase = []
for name, mask in [
    ('correct battery',   correct & (lab['material'] == M['battery'])),
    ('correct glass',     correct & (lab['material'] == M['glass'])),
    ('correct sharps',    correct & (lab['material'] == M['sharps'])),
    ('haz. chemical',     (lab['material'] == M['hazardous_chemical'])),
    ('composite',         (lab['material'] == M['composite'])),
    ('correct plastic',   correct & (lab['material'] == M['plastic'])),
]:
    it = first_item(mask)
    if it is not None and it not in [s[1] for s in showcase]:
        showcase.append((name, it))
showcase = showcase[:6]
print('showcase items:', showcase)

panels = []
for name, it in showcase:
    r = df_te[df_te.item_id == it].iloc[0]
    x, disp = preprocess_row(r)
    cam_m, zm = gradcam(x, 'material')
    cam_t, _  = gradcam(x, 'threat')
    pm = T.softmax_np(zm[None])[0]
    pred = T.MATERIALS[int(pm.argmax())]; gt = T.MATERIALS[int(r['y_material'])]
    fig, axes = plt.subplots(1, 3, figsize=(8.4, 3.0))
    for ax, im, ttl in [(axes[0], disp / 255.0, f'input (GT: {gt})'),
                        (axes[1], overlay(disp, cam_m), f'material CAM (pred: {pred} {pm.max():.2f})'),
                        (axes[2], overlay(disp, cam_t), 'threat CAM')]:
        ax.imshow(im); ax.set_title(ttl, fontsize=9); ax.axis('off')
    fig.tight_layout()
    fp = f'{OUT}/gradcam_{it}.png'
    fig.savefig(fp, dpi=200, bbox_inches='tight'); plt.close(fig)
    panels.append((name, it, disp, cam_m, cam_t, pred, gt, float(pm.max())))
    print('wrote', fp)

# combined grid for the report (rows = items; cols = input / material CAM / threat CAM)
n = len(panels)
fig, axes = plt.subplots(n, 3, figsize=(8.4, 2.6 * n))
for i, (name, it, disp, cam_m, cam_t, pred, gt, conf) in enumerate(panels):
    row = axes[i] if n > 1 else axes
    row[0].imshow(disp / 255.0); row[0].set_title(f'{it}  (GT: {gt})', fontsize=9)
    row[1].imshow(overlay(disp, cam_m)); row[1].set_title(f'material: {pred} ({conf:.2f})', fontsize=9)
    row[2].imshow(overlay(disp, cam_t)); row[2].set_title('threat evidence', fontsize=9)
    for ax in row: ax.axis('off')
fig.tight_layout()
fig.savefig(f'{OUT}/gradcam_grid.png', dpi=200, bbox_inches='tight'); plt.close(fig)
print('wrote gradcam_grid.png')

## Outputs
`gradcam_<item>.png` (one per showcase item) and `gradcam_grid.png` - download into
`Results/SC21/`. The grid slots into a short
'Qualitative Analysis / Explainability' subsection of the Results chapter.